# Multi-Crop Leaf Disease Detection - Kaggle Training + Quantization (Kaggle Dataset)

**This notebook trains MobileNetV2 and EfficientNetB0 models and exports TFLite (dynamic + full INT8) using a Kaggle Dataset for input and Kaggle Working storage for outputs.**

**Prerequisites:**
- Notebook Settings -> Accelerator -> GPU
- Add your dataset in the right panel (Add data)
- Dataset folder contains `processed/train`, `processed/val`, `processed/test`

**Key differences (Kaggle):**
- Dataset is read from `/kaggle/input/<dataset-name>` (read-only)
- Models and TFLite exports are saved to `/kaggle/working`
- Use `Save Version` to export results as a Kaggle Dataset

## Step 1: Set Kaggle dataset path

In [ ]:
import os

print('Available datasets in /kaggle/input:')
print(os.listdir('/kaggle/input'))

# TODO: change to your Kaggle dataset slug
KAGGLE_DATASET = '/kaggle/input/datasets/mdhasibultamim/multi-crop-leaf-disease/processed'
DATASET_BASE = os.path.join(KAGGLE_DATASET, 'processed')

if os.path.exists(DATASET_BASE):
    print('Found dataset:', DATASET_BASE)
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(DATASET_BASE, split)
        if os.path.exists(split_path):
            n_classes = len([
                d for d in os.listdir(split_path)
                if os.path.isdir(os.path.join(split_path, d))
            ])
            print(f'  OK {split}: {n_classes} classes')
        else:
            print(f'  MISSING {split}')
else:
    print('Dataset not found:', DATASET_BASE)
    print('Update KAGGLE_DATASET to your dataset slug.')

## Step 2: Optional - copy dataset to /kaggle/working (faster IO)

In [ ]:
import os

if "DATASET_BASE" not in globals():
    raise ValueError("DATASET_BASE not set. Run Step 1 first.")

source_dataset = DATASET_BASE

if os.path.exists(source_dataset):
    if os.path.exists(os.path.join(source_dataset, "train")):
        dataset_base = source_dataset
    else:
        raise ValueError(f"Expected train/val/test folders under {source_dataset}")
    print("Using dataset:", dataset_base)
else:
    print("Source dataset not found:", source_dataset)

## Step 3: Clone the Complete Project Repository

In [ ]:
# Kaggle Settings -> Internet must be enabled for this one-time clone.
import os
import subprocess

REPO_URL = 'https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git'
repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
required_paths = [
    'requirements.txt',
    'training/train.py',
    'training/evaluate.py',
    'quantization/post_training_quant.py',
    'quantization/evaluate_tflite.py',
    'quantization/qat.py',
    'training/config_efficientnet_b0.yaml',
]

repo_is_complete = all(
    os.path.exists(os.path.join(repo_dir, path)) for path in required_paths
)
if not os.path.exists(repo_dir):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, repo_dir], check=True)
elif not repo_is_complete:
    raise RuntimeError(
        'The existing repo directory is incomplete. Restart the Kaggle session or '
        'remove only /kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System, then rerun this cell.'
    )
else:
    print('Complete repository already exists:', repo_dir)

if not all(os.path.exists(os.path.join(repo_dir, path)) for path in required_paths):
    raise FileNotFoundError('Repository clone is missing a required training file.')

print('Repository ready:', repo_dir)

In [ ]:
import os

repo_dir = "/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
if not os.path.exists(repo_dir):
    raise FileNotFoundError(
        "Repo not found. Run the repository clone cell first."
    )

%cd {repo_dir}
print("Repository ready!")

## Step 4: Install Dependencies

In [ ]:
%pip install -q -r requirements.txt
%pip install -q tensorflow-model-optimization
print('Dependencies installed successfully!')

In [ ]:
import os
import shutil

os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TFHUB_CACHE_DIR"] = "/kaggle/working/tfhub_modules"
shutil.rmtree(os.environ["TFHUB_CACHE_DIR"], ignore_errors=True)
os.makedirs(os.environ["TFHUB_CACHE_DIR"], exist_ok=True)
print("TF_USE_LEGACY_KERAS:", os.environ["TF_USE_LEGACY_KERAS"])
print("TF Hub cache:", os.environ["TFHUB_CACHE_DIR"])

**Preprocessing note:** MobileNetV2 uses `mobilenet_v2.preprocess_input` (`[-1, 1]`). EfficientNetB0 uses the Keras EfficientNet input contract (raw `0..255` values; the model includes rescaling). Optional TF Hub EfficientNet-Lite0 uses `[0, 1]`. Make sure deployment uses the same normalization for the selected model.

## Step 5: Configure Training for MobileNetV2 (2-Phase Schedule)

In [ ]:
import os
import yaml

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
cfg_path = os.path.join(repo_dir, 'training', 'config_mobilenetv2.yaml')

if 'dataset_base' not in globals():
    dataset_base = DATASET_BASE

output_base = '/kaggle/working/leaf_models'
os.makedirs(output_base, exist_ok=True)

with open(cfg_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['data_dir'] = dataset_base
cfg['dataset']['train_dir'] = '{}/train'.format(dataset_base)
cfg['dataset']['val_dir'] = '{}/val'.format(dataset_base)
cfg['dataset']['test_dir'] = '{}/test'.format(dataset_base)

train_dir = cfg['dataset']['train_dir']
if not os.path.exists(train_dir):
    raise FileNotFoundError('Train directory not found: {}'.format(train_dir))

num_classes = len([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])
cfg['model']['num_classes'] = num_classes

cfg['training']['epochs'] = 25
cfg['training']['freeze_base'] = True
cfg['training']['unfreeze_epoch'] = 5
cfg['training']['freeze_until_layer'] = 50
cfg['training']['fine_tune_learning_rate'] = 0.0001

cfg['optimizer']['learning_rate'] = 0.001
cfg['lr_schedule']['type'] = 'reduce_on_plateau'
cfg['lr_schedule']['monitor'] = 'val_loss'
cfg['lr_schedule']['factor'] = 0.5
cfg['lr_schedule']['patience'] = 5
cfg['lr_schedule']['min_lr'] = 1e-7

cfg['callbacks']['early_stopping']['patience'] = 5

cfg['export']['save_dir'] = output_base
cfg['export']['sync_flutter_labels'] = False

cfg['callbacks']['tensorboard']['enabled'] = False
cfg['callbacks']['csv_logger']['filename'] = '{}/training_log_mobilenetv2.csv'.format(output_base)

with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Configuration updated for Kaggle')
print('Training schedule: 5 epochs frozen + 20 epochs fine-tune')
print('Unfreeze from layer: 50')
print('Classes: {}'.format(cfg['model']['num_classes']))
print('Train dataset: {}'.format(cfg['dataset']['train_dir']))
print('Models save to: {}'.format(cfg['export']['save_dir']))
print('TensorBoard logs: DISABLED (saves space)')

## Step 6: Train MobileNetV2

In [ ]:
%cd /kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System/training

!python train.py --config config_mobilenetv2.yaml

## Step 7: Configure EfficientNetB0 (2-Phase Schedule)

In [ ]:
import os
import yaml

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
cfg_path = os.path.join(repo_dir, 'training', 'config_efficientnet_b0.yaml')

if 'dataset_base' not in globals():
    dataset_base = DATASET_BASE

output_base = '/kaggle/working/leaf_models'
os.makedirs(output_base, exist_ok=True)

with open(cfg_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['data_dir'] = dataset_base
cfg['dataset']['train_dir'] = '{}/train'.format(dataset_base)
cfg['dataset']['val_dir'] = '{}/val'.format(dataset_base)
cfg['dataset']['test_dir'] = '{}/test'.format(dataset_base)

train_dir = cfg['dataset']['train_dir']
if not os.path.exists(train_dir):
    raise FileNotFoundError('Train directory not found: {}'.format(train_dir))

num_classes = len([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])
cfg['model']['num_classes'] = num_classes

cfg['training']['epochs'] = 25
cfg['training']['freeze_base'] = True
cfg['training']['unfreeze_epoch'] = 5
cfg['training']['freeze_until_layer'] = 50
cfg['training']['fine_tune_learning_rate'] = 0.0001

cfg['optimizer']['learning_rate'] = 0.001
cfg['lr_schedule']['type'] = 'reduce_on_plateau'
cfg['lr_schedule']['monitor'] = 'val_loss'
cfg['lr_schedule']['factor'] = 0.5
cfg['lr_schedule']['patience'] = 5
cfg['lr_schedule']['min_lr'] = 1e-7

cfg['callbacks']['early_stopping']['patience'] = 5

cfg['export']['save_dir'] = output_base
cfg['export']['sync_flutter_labels'] = False

cfg['callbacks']['tensorboard']['enabled'] = False
cfg['callbacks']['csv_logger']['filename'] = '{}/training_log_efficientnet_b0.csv'.format(output_base)

with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('EfficientNetB0 config updated for Kaggle')
print('Training schedule: 5 epochs frozen + 20 epochs fine-tune')
print('Unfreeze from layer: 50')
print('Classes: {}'.format(cfg['model']['num_classes']))
print('Models save to: {}'.format(cfg['export']['save_dir']))

## Step 8: Train EfficientNetB0

In [ ]:
%cd /kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System/training

!python train.py --config config_efficientnet_b0.yaml

## Step 9: List Trained Models and Exports

In [ ]:
import os

models_dir = '/kaggle/working/leaf_models'

print('Trained models saved to /kaggle/working:\n')

for arch in ['mobilenetv2', 'efficientnet_b0']:
    arch_dir = os.path.join(models_dir, arch)
    if os.path.exists(arch_dir):
        print('{}:'.format(arch))
        for f in sorted(os.listdir(arch_dir)):
            fpath = os.path.join(arch_dir, f)
            if os.path.isfile(fpath):
                fsize = os.path.getsize(fpath) / (1024**2)
                print('  - {} ({:.1f} MB)'.format(f, fsize))
            else:
                print('  - {}/ (dir)'.format(f))
    else:
        print('{}: No models found yet'.format(arch))

## Step 10: Evaluate One Trained Model (Optional)

In [ ]:
import os
import glob
import subprocess

# Choose architecture to evaluate
arch = 'mobilenetv2'  # or 'efficientnet_b0'
models_dir = '/kaggle/working/leaf_models/{}'.format(arch)

def find_latest_model(arch_dir):
    saved_models = sorted(
        glob.glob(os.path.join(arch_dir, 'saved_model_*')),
        key=os.path.getmtime,
    )
    if saved_models:
        return saved_models[-1]
    h5_models = sorted(
        glob.glob(os.path.join(arch_dir, '*.h5')),
        key=os.path.getmtime,
    )
    if h5_models:
        return h5_models[-1]
    return None

if os.path.exists(models_dir):
    model_path = find_latest_model(models_dir)
    if model_path:
        print('Evaluating: {}\n'.format(os.path.basename(model_path)))
        training_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System/training'
        os.chdir(training_dir)
        config_name = (
            'config_mobilenetv2.yaml'
            if arch == 'mobilenetv2'
            else 'config_efficientnet_b0.yaml'
        )
        subprocess.run(['python', 'evaluate.py', '--model', model_path, '--config', config_name])
    else:
        print('No model found for evaluation')
else:
    print('Directory not found: {}'.format(models_dir))

## Step 11: Evaluate All Trained Models

In [ ]:
import glob
import json
import os
import subprocess
import pandas as pd

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
models_root = '/kaggle/working/leaf_models'
evaluation_root = os.path.join(models_root, 'evaluations', 'trained')
os.makedirs(evaluation_root, exist_ok=True)

def find_latest_trained_model(arch_dir):
    saved_models = sorted(glob.glob(os.path.join(arch_dir, 'saved_model_*')), key=os.path.getmtime)
    if saved_models:
        return saved_models[-1]
    h5_models = sorted(glob.glob(os.path.join(arch_dir, '*.h5')), key=os.path.getmtime)
    return h5_models[-1] if h5_models else None

summaries = []
for arch in ['mobilenetv2', 'efficientnet_b0']:
    model_path = find_latest_trained_model(os.path.join(models_root, arch))
    if not model_path:
        print('{}: no trained model found'.format(arch))
        continue

    report_dir = os.path.join(evaluation_root, arch)
    config_path = os.path.join(repo_dir, 'training', 'config_{}.yaml'.format(arch))
    print('Evaluating {}: {}'.format(arch, os.path.basename(model_path)))
    command = [
        'python', os.path.join(repo_dir, 'training', 'evaluate.py'),
        '--model', model_path,
        '--config', config_path,
        '--results-dir', report_dir,
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('Evaluation failed for {}'.format(model_path))

    with open(os.path.join(report_dir, 'metrics_summary.json')) as handle:
        summary = json.load(handle)
    summary['architecture'] = arch
    summaries.append(summary)

if summaries:
    summary_path = os.path.join(evaluation_root, 'model_comparison.csv')
    comparison = pd.DataFrame(summaries).sort_values('accuracy', ascending=False)
    comparison.to_csv(summary_path, index=False)
    display(comparison)
    print('Saved trained-model comparison:', summary_path)


## Step 11: Quantize Models (Dynamic Range)

In [ ]:
import os
import glob
import subprocess

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
quant_script = os.path.join(repo_dir, 'quantization', 'post_training_quant.py')
models_root = '/kaggle/working/leaf_models'
output_dir = '/kaggle/working/leaf_models/quantized'
os.makedirs(output_dir, exist_ok=True)

def find_latest_model(arch_dir):
    saved_models = sorted(
        glob.glob(os.path.join(arch_dir, 'saved_model_*')),
        key=os.path.getmtime,
    )
    if saved_models:
        return saved_models[-1]
    h5_models = sorted(
        glob.glob(os.path.join(arch_dir, '*.h5')),
        key=os.path.getmtime,
    )
    if h5_models:
        return h5_models[-1]
    weights_models = sorted(
        glob.glob(os.path.join(arch_dir, 'checkpoints', '*.weights.h5')),
        key=os.path.getmtime,
    )
    if weights_models:
        return weights_models[-1]
    return None

for arch in ['mobilenetv2', 'efficientnet_b0']:
    arch_dir = os.path.join(models_root, arch)
    model_path = find_latest_model(arch_dir)
    if not model_path:
        print('{}: No model found for quantization'.format(arch))
        continue
    out_path = os.path.join(output_dir, '{}_dynamic.tflite'.format(arch))
    config_path = os.path.join(repo_dir, 'training', 'config_{}.yaml'.format(arch))
    print('Quantizing (dynamic range): {}'.format(arch))
    cmd = [
        'python',
        quant_script,
        '--model_path',
        model_path,
        '--output_path',
        out_path,
        '--arch',
        arch,
    ]
    if model_path.endswith('.weights.h5'):
        cmd += ['--config', config_path]
    subprocess.run(cmd, check=False)

## Step 12: Quantize Models (Full INT8)

In [ ]:
import os
import glob
import subprocess

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
quant_script = os.path.join(repo_dir, 'quantization', 'post_training_quant.py')
models_root = '/kaggle/working/leaf_models'
output_dir = '/kaggle/working/leaf_models/quantized'

if 'dataset_base' not in globals():
    dataset_base = DATASET_BASE

rep_dir = '{}/train'.format(dataset_base)
test_dir = '{}/test'.format(dataset_base)
os.makedirs(output_dir, exist_ok=True)

def find_latest_model(arch_dir):
    saved_models = sorted(
        glob.glob(os.path.join(arch_dir, 'saved_model_*')),
        key=os.path.getmtime,
    )
    if saved_models:
        return saved_models[-1]
    h5_models = sorted(
        glob.glob(os.path.join(arch_dir, '*.h5')),
        key=os.path.getmtime,
    )
    if h5_models:
        return h5_models[-1]
    weights_models = sorted(
        glob.glob(os.path.join(arch_dir, 'checkpoints', '*.weights.h5')),
        key=os.path.getmtime,
    )
    if weights_models:
        return weights_models[-1]
    return None

for arch in ['mobilenetv2', 'efficientnet_b0']:
    arch_dir = os.path.join(models_root, arch)
    model_path = find_latest_model(arch_dir)
    if not model_path:
        print('{}: No model found for INT8 quantization'.format(arch))
        continue
    out_path = os.path.join(output_dir, '{}_int8.tflite'.format(arch))
    config_path = os.path.join(repo_dir, 'training', 'config_{}.yaml'.format(arch))
    print('Quantizing (full INT8): {}'.format(arch))
    cmd = [
        'python',
        quant_script,
        '--model_path',
        model_path,
        '--output_path',
        out_path,
        '--representative_data',
        rep_dir,
        '--evaluate',
        '--test_data',
        test_dir,
        '--arch',
        arch,
    ]
    if model_path.endswith('.weights.h5'):
        cmd += ['--config', config_path]
    subprocess.run(cmd, check=False)

## Step 13: Benchmark TFLite Models (Kaggle CPU)

In [ ]:
import os
import time
import numpy as np
import tensorflow as tf

quant_dir = '/kaggle/working/leaf_models/quantized'

if not os.path.exists(quant_dir):
    print('Quantized models directory not found: {}'.format(quant_dir))
else:
    tflite_files = [f for f in os.listdir(quant_dir) if f.endswith('.tflite')]
    if not tflite_files:
        print('No .tflite files found in {}'.format(quant_dir))
    else:
        def benchmark_tflite(path, runs=100):
            interpreter = tf.lite.Interpreter(model_path=path)
            interpreter.allocate_tensors()
            input_details = interpreter.get_input_details()
            input_shape = input_details[0]['shape']
            input_dtype = input_details[0]['dtype']

            if input_dtype == np.uint8:
                dummy = np.random.randint(0, 255, size=input_shape, dtype=np.uint8)
            else:
                dummy = np.random.rand(*input_shape).astype(np.float32)

            for _ in range(10):
                interpreter.set_tensor(input_details[0]['index'], dummy)
                interpreter.invoke()

            times = []
            for _ in range(runs):
                start = time.perf_counter()
                interpreter.set_tensor(input_details[0]['index'], dummy)
                interpreter.invoke()
                times.append((time.perf_counter() - start) * 1000)

            avg = float(np.mean(times))
            std = float(np.std(times))
            print('{}: {:.2f} +/- {:.2f} ms'.format(os.path.basename(path), avg, std))

        print('Benchmarking TFLite models (CPU):')
        for name in sorted(tflite_files):
            benchmark_tflite(os.path.join(quant_dir, name))

## Step 14: Evaluate Optimized TFLite Models

In [ ]:
import glob
import json
import os
import subprocess
import pandas as pd

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
models_root = '/kaggle/working/leaf_models'
quant_dir = os.path.join(models_root, 'quantized')
evaluation_root = os.path.join(models_root, 'evaluations', 'tflite')
test_dir = '{}/test'.format(dataset_base if 'dataset_base' in globals() else DATASET_BASE)
os.makedirs(evaluation_root, exist_ok=True)

summaries = []
for model_path in sorted(glob.glob(os.path.join(quant_dir, '*.tflite'))):
    filename = os.path.basename(model_path)
    arch = 'efficientnet_b0' if 'efficientnet_b0' in filename.lower() else 'mobilenetv2'
    report_dir = os.path.join(evaluation_root, os.path.splitext(filename)[0])
    print('Evaluating {}...'.format(filename))
    command = [
        'python', os.path.join(repo_dir, 'quantization', 'evaluate_tflite.py'),
        '--model', model_path,
        '--test-data', test_dir,
        '--output-dir', report_dir,
        '--arch', arch,
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('TFLite evaluation failed for {}'.format(model_path))

    with open(os.path.join(report_dir, 'metrics_summary.json')) as handle:
        summary = json.load(handle)
    summary['model_file'] = filename
    summaries.append(summary)

if summaries:
    summary_path = os.path.join(evaluation_root, 'tflite_comparison.csv')
    comparison = pd.DataFrame(summaries).sort_values('accuracy', ascending=False)
    comparison.to_csv(summary_path, index=False)
    display(comparison)
    print('Saved TFLite comparison:', summary_path)
else:
    print('No TFLite models found in {}'.format(quant_dir))


## Step 15: Quantization-Aware Training (EfficientNetB0)

In [ ]:
import glob
import json
import os
import subprocess
import sys
import pandas as pd

try:
    import tensorflow_model_optimization as tfmot
    print('TFMOT version:', getattr(tfmot, '__version__', 'unknown'))
except Exception as exc:
    raise RuntimeError(
        'QAT requires tensorflow-model-optimization. If Kaggle Python 3.12 '
        'cannot import it, use the existing PTQ cells or a compatible TensorFlow environment.'
    ) from exc

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
models_root = '/kaggle/working/leaf_models'
arch = 'efficientnet_b0'
config_path = os.path.join(repo_dir, 'training', 'config_{}.yaml'.format(arch))
qat_script = os.path.join(repo_dir, 'quantization', 'qat.py')
test_dir = '{}/test'.format(dataset_base if 'dataset_base' in globals() else DATASET_BASE)
representative_dir = '{}/train'.format(dataset_base if 'dataset_base' in globals() else DATASET_BASE)
qat_root = os.path.join(models_root, 'qat', arch)
os.makedirs(qat_root, exist_ok=True)

def find_qat_source(arch_dir):
    checkpoints = sorted(glob.glob(os.path.join(arch_dir, 'checkpoints', '*.weights.h5')), key=os.path.getmtime)
    if checkpoints:
        return checkpoints[-1]
    saved_models = sorted(glob.glob(os.path.join(arch_dir, 'saved_model_*')), key=os.path.getmtime)
    return saved_models[-1] if saved_models else None

def run_checked(command):
    result = subprocess.run(command, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('Command failed: {}'.format(' '.join(command)))

source_model = find_qat_source(os.path.join(models_root, arch))
if not source_model:
    raise FileNotFoundError('Run EfficientNetB0 training before QAT.')

run_checked([
    sys.executable, qat_script,
    '--model_path', source_model,
    '--config', config_path,
    '--output_dir', qat_root,
    '--representative_data', representative_dir,
    '--arch', arch,
    '--epochs', '8',
    '--learning_rate', '1e-5',
    '--qat_mode', 'auto',
])

qat_model_path = os.path.join(qat_root, 'qat_stripped.h5')
qat_tflite_path = os.path.join(qat_root, 'qat_int8.tflite')
qat_float_report = os.path.join(qat_root, 'evaluation_float')
qat_tflite_report = os.path.join(qat_root, 'evaluation_tflite')
run_checked([
    sys.executable, os.path.join(repo_dir, 'training', 'evaluate.py'),
    '--model', qat_model_path, '--config', config_path,
    '--results-dir', qat_float_report,
])
run_checked([
    sys.executable, os.path.join(repo_dir, 'quantization', 'evaluate_tflite.py'),
    '--model', qat_tflite_path, '--test-data', test_dir,
    '--output-dir', qat_tflite_report, '--arch', arch,
])

baseline_path = os.path.join(models_root, 'evaluations', 'trained', arch, 'metrics_summary.json')
qat_float_path = os.path.join(qat_float_report, 'metrics_summary.json')
qat_tflite_metrics_path = os.path.join(qat_tflite_report, 'metrics_summary.json')
if not os.path.exists(baseline_path):
    raise FileNotFoundError('Run Step 11 baseline evaluation before comparing QAT.')

with open(baseline_path) as handle:
    baseline = json.load(handle)
with open(qat_float_path) as handle:
    qat_float = json.load(handle)
with open(qat_tflite_metrics_path) as handle:
    qat_tflite = json.load(handle)

comparison = pd.DataFrame([
    {'model': 'float_baseline', **baseline},
    {'model': 'qat_float', **qat_float},
    {'model': 'qat_int8_tflite', **qat_tflite},
])
comparison_path = os.path.join(qat_root, 'qat_comparison.csv')
comparison.to_csv(comparison_path, index=False)
display(comparison)

accuracy_drop_pp = (baseline['accuracy'] - qat_tflite['accuracy']) * 100.0
macro_f1_drop_pp = (baseline['macro_f1'] - qat_tflite['macro_f1']) * 100.0
accepted = accuracy_drop_pp <= 0.5 and macro_f1_drop_pp <= 1.0
decision = {
    'accepted': bool(accepted),
    'accuracy_drop_percentage_points': float(accuracy_drop_pp),
    'macro_f1_drop_percentage_points': float(macro_f1_drop_pp),
    'max_accuracy_drop_percentage_points': 0.5,
    'max_macro_f1_drop_percentage_points': 1.0,
    'comparison_csv': comparison_path,
}
with open(os.path.join(qat_root, 'qat_decision.json'), 'w') as handle:
    json.dump(decision, handle, indent=2)
print(json.dumps(decision, indent=2))


## Step 16: View Training Logs (Kaggle Working)

In [ ]:
import os
import pandas as pd

results_dir = '/kaggle/working/leaf_models'

if os.path.exists(results_dir):
    csv_files = [f for f in os.listdir(results_dir) if f.endswith('.csv')]
    if csv_files:
        for csv_file in csv_files:
            csv_path = os.path.join(results_dir, csv_file)
            print('\n=== {} ==='.format(csv_file))
            df = pd.read_csv(csv_path)
            print(df.tail(10))
    else:
        print('No CSV files found yet')
else:
    print('Results directory not found yet')

## Step 17: Export Trained Models (Kaggle Files / Save Version)

In [ ]:
import os
import shutil

models_dir = '/kaggle/working/leaf_models'
zip_base = '/kaggle/working/leaf_models'
zip_path = '{}.zip'.format(zip_base)

print('=' * 60)
print('EXPORT TRAINED MODELS')
print('=' * 60)

if os.path.exists(models_dir):
    if os.path.exists(zip_path):
        os.remove(zip_path)
    print('Preparing models archive...')
    shutil.make_archive(zip_base, 'zip', models_dir)
    print('Archive created: {}'.format(zip_path))
    print('Use Save Version to export this zip as a dataset.')
else:
    print('No models found to export')

## Storage Summary

**What happened:**
- Dataset read from Kaggle input (`/kaggle/input/<dataset-name>/processed`)
- Models trained and saved to `/kaggle/working/leaf_models/`
- Quantized TFLite models saved to `/kaggle/working/leaf_models/quantized/`
- QAT models and reports saved to `/kaggle/working/leaf_models/qat/efficientnet_b0/`
- Optional export as `leaf_models.zip` in `/kaggle/working`

**Kaggle storage notes:**
- `/kaggle/input` is read-only
- `/kaggle/working` is writable (saved with the notebook run)
- Use Save Version to publish outputs as a Kaggle Dataset

**Next steps:**
1. Save a version to keep the trained models
2. Download `leaf_models.zip` from the notebook output or Kaggle Files
3. Use the INT8 TFLite model in your Flutter app for best on-device speed